# 1. Úkoly

## 1.1 Základní maticové násobení

Pomocí NumPy a Numby napište násobení dvou čtvercových matic.

- Výsledek ukládejte do předem připravené matice `C`, tedy funkce bude mít tvar `matrix_multiplication(A, B, C)`.
- Zakompilujte funkci a pro matice `1000 x 1000` změřte čas výpočtu.
- Porovnejte výsledek i čas s `numpy.dot`.

In [1]:
import numpy as np
from numba import njit, prange
import time

In [2]:
@njit
def matrix_multiplication(A, B, C):
    n = A.shape[0]
    for i in range(n):
        for j in range(n):
            temp = 0.0
            for k in range(n):
                temp += A[i, k] * B[k, j]
            C[i, j] = temp

## 1.2 Chunk režim

Upravte kód tak, aby výpočet běžel po blocích (chunk mode).

- Použijte celkem 6 vnořených smyček:
  - 3 vnější přes chunky v osách `i`, `j`, `k`,
  - 3 vnitřní přes prvky uvnitř každého chunku.
- Zakompilujte, změřte čas a porovnejte výsledek s bodem 1 i s `numpy.dot`.

In [3]:
@njit
def matrix_multiplication_chunk(A, B, C, chunk_size):
    n = A.shape[0]
    for i in range(n):
        for j in range(n):
            C[i, j] = 0.0
            
    for i_chunk in range(0, n, chunk_size):
        for j_chunk in range(0, n, chunk_size):
            for k_chunk in range(0, n, chunk_size):
                
                for i in range(i_chunk, min(i_chunk + chunk_size, n)):
                    for j in range(j_chunk, min(j_chunk + chunk_size, n)):
                        temp = 0.0
                        for k in range(k_chunk, min(k_chunk + chunk_size, n)):
                            temp += A[i, k] * B[k, j]
                        C[i, j] += temp

## 1.3 Paralelizace

Upravte funkci tak, aby výpočet běžel paralelně. Otestujte, jaký vliv má volba paralelní vnější smyčky.

- Porovnejte výsledek i čas s NumPy a s řešením z bodů 1 a 2.

In [4]:
@njit(parallel=True)
def matrix_multiplication_parallel(A, B, C):
    n = A.shape[0]
    for i in prange(n):
        for j in range(n):
            temp = 0.0
            for k in range(n):
                temp += A[i, k] * B[k, j]
            C[i, j] = temp

In [5]:
def main():
    N = 1000
    chunk_size = 64
    
    np.random.seed(42)
    A = np.random.rand(N, N)
    B = np.random.rand(N, N)
    
    C_basic = np.zeros((N, N))
    C_chunk = np.zeros((N, N))
    C_parallel = np.zeros((N, N))

    print(f"Velikost matic: {N} x {N}\n")

    start_time = time.time()
    C_numpy = np.dot(A, B)
    numpy_time = time.time() - start_time
    print(f"numpy.dot cas: {numpy_time:.4f} s")

    matrix_multiplication(A[:10, :10], B[:10, :10], np.zeros((10, 10))) 
    start_time = time.time()
    matrix_multiplication(A, B, C_basic)
    basic_time = time.time() - start_time
    print(f"1.1 Zakladni nasobeni (Numba) cas: {basic_time:.4f} s")
    print(f"Shoda s NumPy: {np.allclose(C_numpy, C_basic)}\n")

    matrix_multiplication_chunk(A[:10, :10], B[:10, :10], np.zeros((10, 10)), 2) 
    start_time = time.time()
    matrix_multiplication_chunk(A, B, C_chunk, chunk_size)
    chunk_time = time.time() - start_time
    print(f"1.2 Chunk rezim cas: {chunk_time:.4f} s")
    print(f"Shoda s NumPy: {np.allclose(C_numpy, C_chunk)}\n")

    matrix_multiplication_parallel(A[:10, :10], B[:10, :10], np.zeros((10, 10))) 
    start_time = time.time()
    matrix_multiplication_parallel(A, B, C_parallel)
    parallel_time = time.time() - start_time
    print(f"1.3 Paralelni rezim cas: {parallel_time:.4f} s")
    print(f"Shoda s NumPy: {np.allclose(C_numpy, C_parallel)}\n")

if __name__ == "__main__":
    main()

Velikost matic: 1000 x 1000

numpy.dot cas: 0.0362 s
1.1 Zakladni nasobeni (Numba) cas: 2.4009 s
Shoda s NumPy: True

1.2 Chunk rezim cas: 2.3880 s
Shoda s NumPy: True

1.3 Paralelni rezim cas: 1.2664 s
Shoda s NumPy: True

